In [ ]:
using Pkg
Pkg.activate("..")

In [ ]:
# Global control variable for bang-bang control (±1)
global α = 1

# Condition function: triggers callback when p = 0 (control switching)
function condition(z, t, integrator)                        
    x⁰,x,p⁰,p = z
    return p
end

# Affect function: flips control sign when condition is met (bang-bang switching)
function affect!(integrator)                                
    global α = -α
    nothing
end

cb = ContinuousCallback(condition, affect!)                     # Continuous callback for control switching
H(x, p) = p[1] * x[2] + α * p[2]                                # Hamiltonian function
ϕ_ = Flow(OptimalControl.Hamiltonian(H), callback = cb)         # Hamiltonian flow with maximizing control 

# Flow wrapper for shooting function: sets initial control direction based on costate
function ϕ(t0, x0, p0, tf; kwargs...)                       
    if p0[2] == 0
        global α = -sign(p0[1])  # Set control based on p⁰ if p₀ = 0
    else
        global α = sign(p0[2])    # Set control based on p
    end
    return ϕ_(t0, x0, p0, tf; kwargs...)
end

# Flow wrapper for plotting: handles time interval tuple
function ϕ((t0, tf), x0, p0; kwargs...)                     
    if p0[2] == 0
        global α = -sign(p0[1])
    else
        global α = sign(p0[2])
    end
    return ϕ_((t0, tf), x0, p0; kwargs...)
end

In [ ]:
# Problem parameters
t0 = 0                                                                      # Initial time
x0 = [0,0]                                                                  # Initial augmented state (x⁰, x)
tf = 5                                                                      # Final time
global xT = 0                                                               # Target final state

# Helper functions
π((x,p)) = x[2]                                                             # Projection on state space (extract x from augmented state)
η(p0) = -sqrt.(1 - p0.^2)                                                   # Function η(⋅) = -√(1 - p0²)

# Shooting function S
S(p0; xT=xT) = π( ϕ(t0, x0, p0, tf) ) - xT                                  # General shooting function

# Normalized shooting functions for different boundary conditions
S₁(p0; xT=xT) = S([-1, p0]; xT=xT)                                          # Normalization 1: fix p⁰ = -1
S₂(p0; xT=xT) = abs(p0) < 1 ? S([η(p0), p0]; xT=xT) : sign(p0)*tf - xT      # Normalization 2: use function η to have |p|=1

# Create animated 3D plot of shooting functions
plt = plot_shooting(S, S₁, S₂, :S)                                          # Plot (see utils.jl)
@gif for i ∈ [range(30, 90, 50); 90*ones(25); range(90, 30, 50); 30*ones(25)]
    plot!(plt[1], camera=(i,i),                                             # Rotate camera angle
        zticks = i==90 ? false : true,
        zlabel = i==90 ? "" : "S" )
end

In [ ]:
# Global vector to store solver iterates
global iterate_S2 = Vector{Float64}()                      

# Wrapper function for S₂ compatible with MINPACK solver (stores iterates)
function S₂!(s₂, ξ; xT=xT)                                         
    push!(iterate_S2, ξ[1])                                 # Store current iterate
    return (s₂[:] .= S₂(ξ[1]; xT=xT); nothing)              # Compute S₂ value in-place                     
end

# Jacobian of S₂ using automatic differentiation
JS₂(ξ; xT=xT) = ForwardDiff.jacobian(p0 -> [S₂(p0[1]; xT=xT)], ξ)         
# Wrapper for Jacobian compatible with MINPACK solver
JS₂!(js₂, ξ; xT=xT) = (js₂[:] .= JS₂(ξ; xT=xT); nothing)                  

ξ = [-0.5]                                                  # Initial guess for solver
p0_sol = fsolve(S₂!, JS₂!, ξ, show_trace = true)            # Solve S₂(p0) = 0 using MINPACK

In [ ]:
# Compute optimal trajectory
sol = ϕ((t0, tf), x0, [η(p0_sol.x[1]), p0_sol.x[1]], saveat=range(t0, tf, 500))
# Plot the optimal solution (state, costate, and control) (see utils.jl)
plot_sol(sol)

In [ ]:
function plot_ellipse(a,b,θ,c,φ,x)
    # Generate data for the boundary of the accessible set
    n_ = 100                                                            # Number of points for boundary
    # Create initial costate values along the boundary
    p0_ = [[[-1, i] for i ∈ range(-tf, 0, n_)];                         # Left side (p⁰ = -1)
        [[1, i] for i ∈ range(tf, 0, n_)]]                              # Right side (p⁰ = 1)
    x_ = zeros(2, 2*n_); p_ = zeros(2, 2*n_)                            # Initialize final state and costate arrays
    # Compute flow for each initial costate to get the boundary
    for i in eachindex(p0_)
        x_[:,i], p_[:,i] = ϕ(t0, x0, p0_[i], tf)                        # Compute Hamiltonian flow
    end

    # Generate ellipse points
    β = range(-Base.π, Base.π; length = 100)                            # Angle parameter for ellipse
    # Compute ellipse points: rotate by -Θ, scale by (a,b), shift by center c
    xₑ = r(-θ)*s(a,b)*
        transpose(reduce(hcat,[sin.(β), cos.(β)])).+c                   # Ellipse boundary points

    # Transform coordinates using the function ϕ
    y = φ(x); y_ = φ(x_); yₑ = φ(xₑ)                                    # Apply transformation to all data

    # Plot in original coordinates (x, x⁰)
    plt_x = plot(x_[2,:], x_[1,:], label = nothing)                     # Accessible set boundary
    scatter!(x[2,:], x[1,:], label="Observations", legend = :topleft)   # Data points
    plot!(xₑ[2,:], xₑ[1,:], label = "Fitted ellipse")                   # Fitted ellipse
    plot!(xlim = [-15,15], ylim = [-15,15], xlabel = "x", ylabel = "x⁰")

    # Plot in transformed coordinates (y, y⁰)
    plt_y = plot(y_[2,:], y_[1,:], label = nothing)                     # Transformed boundary
    scatter!(y[2,:], y[1,:], label="")                                  # Transformed observations
    plot!(yₑ[2,:], yₑ[1,:], label = "")                                 # Transformed ellipse
    plot!( xlabel = "y", ylabel = "y⁰")

    return plot(plt_x, plt_y, layout = (1,2), size=(800, 400))
end

In [ ]:
# Generate data points for ellipse fitting
n = 15                                              # Number of sample points
# Create initial costate values along the boundary (both sides)
p0 = [[[-1, i] for i ∈ range(-tf, 0, n)];           # Left side (p⁰ = -1)
      [[1, i] for i ∈ range(tf, 0, n)]]             # Right side (p⁰ = 1)
    
# Compute final states and costates by compute the flow from different initial costates
x = zeros(2, 2*n); p = zeros(2,2*n)                 # Initialize arrays
for i = 1:length(p0)
    x[:,i], p[:,i] = ϕ(t0, x0, p0[i], tf)           # Compute Hamiltonian flow
end

# Fit ellipse to the accessible set boundary (see utils.jl)
a, b, θ, c = fit_ellipse(x[1,:], x[2,:])            # a: semi-major, b: semi-minor, θ: rotation, c: center

# Helper matrices for coordinate transformation
r(β) = [[cos(β), sin(β)] [-sin(β), cos(β)]]         # 2x2 rotation matrix
s(a,b) = [[a,0] [0,b]]                              # 2x2 scaling matrix

# Construct linear diffeomorphism φ(x) = A*x + B to normalize the ellipse to a circle
d = (a*sin(θ))/(b*cos(θ)); β₀ = atan(d)             # Intermediate values for transformation
A = r(-β₀)*s(1/a,1/b)*r(θ); B = -A*c                # Compute transformation matrices A and B
φ(x) = A*x .+ B                                     # Diffeomorphism ϕ (affine transformation)

# Visualize the fitted ellipse and transformation (see utils.jl)
plot_ellipse(a, b, θ, c, φ, x)

In [ ]:
# Transform shooting functions to new coordinates using the diffeomorphism
p₀(p) = [p[1], p[2] + p[1]*tf]                                              # Inverse transformation for costate
Aₓ = A[2,2]; Bₓ = B[2]                                                      # Extract relevant components for state
yT = Aₓ*xT + Bₓ                                                             # Target state in new coordinates

# Preconditioned shooting function T in new coordinates
T(q; xT=xT) = Aₓ*(S(p₀(transpose(A)*q); xT=xT))                             # General shooting function 
T₁(q; xT=xT) = T([-1, q]; xT=xT)                                            # Normalization 1: fix q⁰ = -1
T₂(q; xT=xT) = abs(q) < 1 ? T([η(q), q]; xT=xT) : sign(q)*(Aₓ*tf + Bₓ)      # Normalization 2: use function η to have |q|=1

# Create animated 3D plot of transformed shooting functions
plt = plot_shooting(T, T₁, T₂, :T)                                          # Plot in T coordinates
@gif for i ∈ [range(30, 90, 50); 90*ones(25); range(90, 30, 50); 30*ones(25)]
    plot!(plt[1], camera=(i,i),                                             # Rotate camera angle
        zticks = i==90 ? false : true,
        zlabel = i==90 ? "" : "T" )
end

In [ ]:
# Global vector to store solver iterates for T₂ analysis
global iterate_T2 = Vector{Float64}()                      

# Wrapper function for T₂ compatible with MINPACK solver (stores iterates)
function T₂!(t₂, ξ; xT=xT)                                         
     push!(iterate_T2, ξ[1])  
     return (t₂[:] .= T₂(ξ[1]; xT=xT); nothing)                   
end

# Jacobian of T₂ using automatic differentiation
JT₂(ξ; xT=xT) = ForwardDiff.jacobian(q0 -> [T₂(q0[1]; xT=xT)], ξ)         
# Wrapper for Jacobian compatible with MINPACK solver
JT₂!(jt₂, ξ; xT=xT) = (jt₂[:] .= JT₂(ξ; xT=xT); nothing)                  

ξ = [0.5]                                                   # Initial guess for solver
q_sol = fsolve(T₂!, JT₂!, ξ, show_trace = true)             # Solve T₂(q) = 0 using MINPACK

In [ ]:
# Transform the solution back to original coordinates
p0_sol = p₀(transpose(A)*[η(q_sol.x[1]), q_sol.x[1]])       # Convert optimal q back to original p coordinates
sol = ϕ((t0, tf), x0, p0_sol, saveat=range(t0, tf, 500))    # Compute optimal trajectory in original coordinates

# Plot the optimal solution
plot_sol(sol)

In [ ]:
# ============================================================================
# CONVERGENCE ANALYSIS: Compare S₂ and T₂ shooting methods
# ============================================================================

# Discretize the accessible final state range
global xT = 0;
q0_span = range(-1, 1, length=10000)                    # Range of q₀ values
T2_span = [T₂(q) for q ∈ q0_span]                       # Compute T₂ over the range
ε = 0.2                                                 # Tolerance for boundary selection
i1 = findfirst(x -> x > T2_span[1]+ε , T2_span)         # Find start index
i2 = findlast(x -> x < T2_span[end]-ε, T2_span)         # Find end index

N = 1000                                                # Number of test points
sol_T2 = range(q0_span[i1], q0_span[i2], length=N)      # q₀ values for testing
# Convert to corresponding p₀ values in original coordinates
sol_S2 = [((p₀(transpose(A)*[η(q), q]))/(norm(p₀(transpose(A)*[η(q), q]), 2)))[2]  for q ∈ sol_T2]

xT_span = [S₂(p) for p ∈ sol_S2]                        # Target states for S₂
yT_span = [T₂(q) for q ∈ sol_T2]                        # Target states for T₂

# Initialize storage arrays for convergence analysis
fnorms_S2 = zeros(N, 100)                               # Norm of S₂ at each iteration
fnorms_T2 = zeros(N, 100)                               # Norm of T₂ at each iteration (in x-space)
fnorms_T2_IG = zeros(N,100)                             # Norm of T₂ at each iteration (in x-space), with natural initial guess
iterates_S2 = zeros(N, 100)                             # Iterates of S₂
iterates_T2 = zeros(N, 100)                             # Iterates of T₂
iterates_T2_IG = zeros(N, 100)                          # Iterates of T₂ with natural initial guess
# Convergence status: -1=not converged, 1=converged, 0=converged but hit bounds
conv_S2 = zeros(N,1)                                    # Convergence status of S₂
conv_T2 = zeros(N,1)                                    # Convergence status of T₂
conv_T2_IG = zeros(N,1)                                 # Convergence status of T₂ with natural initial guess

# Intermediate function: compute S value from T iterates (for error comparison)
T₂_(q0) = abs(q0) < 1 ? S(p₀(transpose(A)*[η.(q0),q0])) : sign(q0) * tf - xT

# Main loop: test convergence for N different target states
for i = 1:N
    # Set current target state
    global xT = xT_span[i]

    ### Test S₂ (original shooting function) ###
    global iterate_S2 = Vector{Float64}()                                               # Clear old iterates
    q_sol_S2 = fsolve(S₂!, JS₂!, [-0.75], show_trace = false, tracing = true)           # Solve with fixed initial guess
    if q_sol_S2.converged
        fnorm_S2 = [q_sol_S2.trace.trace[j].fnorm for j ∈ 1:length(q_sol_S2.trace.trace)]
        iterates_S2[i,1:length(iterate_S2)] = iterate_S2
        conv_S2[i] = length(findall(x-> abs(x) > 1, iterate_S2)) == 0           
        fnorms_S2[i,1:length(fnorm_S2)] = fnorm_S2
    else
        conv_S2[i] = -1                                                                 # Not converged
    end

    ### Test T₂ (transformed shooting function) ###
    global iterate_T2 = Vector{Float64}()                                               # Clear old iterates
    q_sol_T2 = fsolve(T₂!, JT₂!, [0.0], show_trace = false, tracing = true)             # Solve with fixed initial guess
    if q_sol_T2.converged
        iterates_T2[i,1:length(iterate_T2)] = iterate_T2
        conv_T2[i] = length(findall(x-> abs(x) > 1, iterate_T2)) == 0
        fnorms_T2[i, 1:length(iterate_T2)] = abs.(T₂_.(iterate_T2))
    else
        conv_T2[i] = -1                                                                 # Not converged
    end

    ### Test T₂ with natural initial guess ###
    global iterate_T2 = Vector{Float64}()                                               # Clear old iterates
    q_sol_T2_IG = fsolve(T₂!, JT₂!, [yT_span[i]], show_trace = false, tracing = true)   # Solve with target as initial guess
    if q_sol_T2_IG.converged
        iterates_T2_IG[i,1:length(iterate_T2)] = iterate_T2
        conv_T2_IG[i] = length(findall(x-> abs(x) > 1, iterate_T2)) == 0
        fnorms_T2_IG[i, 1:length(iterate_T2)] = abs.(T₂_.(iterate_T2))
    else
        conv_T2_IG[i] = -1                               # Not converged
    end
end

# Print convergence statistics
println("Convergence rates:")
println("S₂: ", 100*(1-length(findall(x -> x == -1, conv_S2))/N), " %")
println("T₂: ", 100*(1-length(findall(x -> x == -1, conv_T2))/N), " %")
println("T₂ with natural initial guess: ", 100*(1-length(findall(x -> x == -1, conv_T2_IG))/N), " %")

# Compute mean error norms across all test cases
mean_fnorms_S2 = mean(fnorms_S2, dims = 1)
mean_fnorms_T2 = mean(fnorms_T2, dims = 1)
mean_fnorms_T2_IG = mean(fnorms_T2_IG, dims = 1)

# Remove trailing zeros (beyond convergence) with tolerance
ε = 1e-9;
mean_fnorms_S2 = mean_fnorms_S2[1:findall(x -> x < ε, mean_fnorms_S2)[1][2]]
mean_fnorms_T2 = mean_fnorms_T2[1:findall(x -> x < ε, mean_fnorms_T2)[1][2]]
mean_fnorms_T2_IG = mean_fnorms_T2_IG[1:findall(x -> x < ε, mean_fnorms_T2_IG)[1][2]]

# Plot 1: Convergence rate comparison (log scale)
plt1 = plot(0:length(mean_fnorms_S2)-1, mean_fnorms_S2, label = "S₂", lw = 3)
plot!(plt1, 0:length(mean_fnorms_T2)-1, mean_fnorms_T2, label = "T₂", lw = 3)
plot!(plt1, 0:length(mean_fnorms_T2_IG)-1, mean_fnorms_T2_IG, label = "T₂ with IG", lw = 3)
plot!(plt1, yaxis = :log10, xlim = [0, 30], ylim = [ε, 10], xlabel = "Iterations", ylabel = "Error")

global xT = 0;

# Plot 2: Convergence regions visualization
plt21 = plot(xlim = [-1,1], xlabel = "p₀", title = "S₂")
plot!(plt21, S₂, c = :black, label = nothing)
# Color code: green=converged, blue=converged but hit bounds, red=not converged
color = [conv_S2[i]==1 ? :green : conv_S2[i] == 0 ? :blue : :red for i ∈ 1:N]
scatter!(sol_S2, xT_span, color = color, markerstrokecolor = color, marker = 2, label =nothing)
plot!(plt21, [-0.75, -0.75], [-5, 5], c = :black, ls = :dash, label = nothing)  # Initial guess line

plt22 = plot(xlim = [-1,1], xlabel = "q₀", title = "T₂")
plot!(plt22, T₂, c = :black, label = nothing)
color = [conv_T2[i]==1 ? :green : conv_T2[i] == 0 ? :blue : :red for i ∈ 1:N]
scatter!(plt22, sol_T2, yT_span, color = color, markerstrokecolor = color, marker = 2, label = "")
plot!(plt22, [0, 0], [T2_span[1], T2_span[end]], c = :black, ls = :dash, label = nothing)  # Initial guess line

plt23 = plot(xlim = [-1,1], xlabel = "q₀", title = "T₂ with IG")
plot!(plt23, T₂, c = :black, label = nothing)
color = [conv_T2_IG[i]==1 ? :green : conv_T2_IG[i] == 0 ? :blue : :red for i ∈ 1:N]
scatter!(plt23, sol_T2, yT_span, color = color, markerstrokecolor = color, marker = 2, label = "")
plot!(plt23, [-1, 1], [-1, 1], c = :black, ls = :dash, label = nothing)  # Natural initial guess line

# Add legend
scatter!(plt21, 1,1, color = :green, markerstrokecolor = :green, marker = 2, label = "converged")
scatter!(plt21, 1,1, color = :blue, markerstrokecolor = :blue, marker = 2, label = "converged but HB")
scatter!(plt21, 1,1, color = :red, markerstrokecolor = :red, marker = 2, label = "not converged")

# Combine plots
plt2 = plot(plt21, plt22, plt23, layout = grid(1,3, widths = [0.30, 0.35, 0.35]))
plot(plt1, plt2, layout = grid(2,1, heights = [0.5, 0.5]), size=(800, 800))